In [20]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import stim
import sinter

REPO = "decoder-bench"
sys.path.insert(0, REPO + "/decoder_bench/ls_ckt_gen")

from circuit_zzxx import circuit as ls_circuit

def coord_union(mapping):
    return (
        set(mapping["data_coords"])
        | set(mapping["x_measure_coords"])
        | set(mapping["z_measure_coords"])
    )

def set_patch_noise(sim, coords, p):
    for coord in coords:
        for gate in sim.gates_1Q:
            sim.profile[coord]["error"][gate] = p
        for gate in sim.gates_2Q:
            sim.profile[coord]["error"][gate] = p
        for gate in sim.measures:
            sim.profile[coord]["error"][gate] = p


def build_ls_stim_circuit(distance, p_left, p_right, basis="Z", bridge_policy="max"):
    sim = ls_circuit(
        distance=distance,
        num_patches_x=20,
        num_patches_y=20,
        spacing=1,
        disable_noise=False,
        fixed_t1=25,
        fixed_t2=40,
        fixed_cnot_latency=50,
        fixed_measure_latency=600,
        fixed_cnot_noise=0.0,
        fixed_measure_noise=0.0,
        rounds_per_op=distance,
        idle_multiplier=1,
        basis=basis,
        ls_basis=basis,
        merge=True,
    )

    left = sim.map_qubit(0)
    right = sim.map_qubit(1)
    merged = sim.merge({0: left, 1: right})

    left_coords = coord_union(left)
    right_coords = coord_union(right)
    bridge_coords = coord_union(merged) - left_coords - right_coords

    if bridge_policy == "max":
        p_bridge = max(p_left, p_right)
    elif bridge_policy == "mean":
        p_bridge = 0.5 * (p_left + p_right)
    elif bridge_policy == "left":
        p_bridge = p_left
    elif bridge_policy == "right":
        p_bridge = p_right
    else:
        raise ValueError("bridge_policy must be one of: max, mean, left, right")

    set_patch_noise(sim, left_coords, p_left)
    set_patch_noise(sim, right_coords, p_right)
    set_patch_noise(sim, bridge_coords, p_bridge)

    sim.from_string("qreg q[2];")

    # sim.ckt is the generated stim circuit text
    return stim.Circuit(sim.ckt)



def make_tasks(
    distances=(3, 5, 7, 9, 11),
    p_values=(1e-4, 2e-4, 3e-4, 5e-4, 7e-4, 1e-3),
    basis="Z",
    right_factor=1.0,
):
    tasks = []
    for p in p_values:
        for d in distances:
            p_left = p
            p_right = right_factor * p

            if np.isclose(right_factor, 1.0):
                mode = "symmetric"
            elif np.isclose(right_factor, 5.0):
                mode = "asymmetric_5x"
            elif np.isclose(right_factor, 10.0):
                mode = "asymmetric_10x"
            else:
                mode = f"asymmetric_{right_factor:g}x"

            c = build_ls_stim_circuit(
                distance=d,
                p_left=p_left,
                p_right=p_right,
                basis=basis,
                bridge_policy="max",
            )

            tasks.append(
                sinter.Task(
                    circuit=c,
                    json_metadata={
                        "mode": mode,
                        "distance": d,
                        "p": p,
                        "p_left": p_left,
                        "p_right": p_right,
                        "right_factor": right_factor,
                    },
                )
            )
    return tasks


def collect_df(tasks, max_shots=20_000, max_errors=400):
    stats = sinter.collect(
        tasks=tasks,
        decoders=["pymatching"],
        max_shots=max_shots,
        max_errors=max_errors,
        num_workers=1,   # change if you want
    )

    rows = []
    for stat in stats:
        md = stat.json_metadata
        rows.append({
            "mode": md["mode"],
            "distance": md["distance"],
            "p": md["p"],
            "p_left": md["p_left"],
            "p_right": md["p_right"],
            "shots": stat.shots,
            "errors": stat.errors,
            "ler": stat.errors / stat.shots if stat.shots else np.nan,
        })
    return pd.DataFrame(rows).sort_values(["mode", "distance", "p"])


In [3]:
distances = [3, 5, 7, 9, 11]
p_values = [1e-4, 3e-4, 7e-4, 1e-3, 3e-3, 7e-3, 1e-2, 3e-2, 7e-2]

tasks_sym = make_tasks(distances=distances, p_values=p_values, right_factor=1.0)
tasks_5x  = make_tasks(distances=distances, p_values=p_values, right_factor=5.0)
tasks_10x = make_tasks(distances=distances, p_values=p_values, right_factor=10.0)

df_sym = collect_df(tasks_sym, max_shots=100_000, max_errors=2_000)
df_5x  = collect_df(tasks_5x,  max_shots=100_000, max_errors=2_000)
df_10x = collect_df(tasks_10x, max_shots=100_000, max_errors=2_000)

# Save simulation data 

In [22]:
import pickle

# Save
with open("sim_data.pkl", "wb") as f:
    pickle.dump({"df_sym": df_sym, "df_5x": df_5x, "df_10x": df_10x}, f)